<a href="https://colab.research.google.com/github/Shea-Fyffe/transforming-personality-item-generation/blob/main/train-dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade accelerate datasets trl peft bitsandbytes transformers[sentencepiece]

In [ ]:
#@title Import Libraries
# standard
import os
import gc
import torch
from dataclasses import (dataclass, field, fields)
from datasets import (Dataset, load_dataset)
import pandas as pd

# application specific
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline)
from peft import (AutoPeftModelForCausalLM, LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training)
from trl import DPOTrainer, DPOConfig
import bitsandbytes as bnb

In [ ]:
#@title Prevent CUDA fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
#@title Model Class
@dataclass
class DPOobj:
    # init arguments
    data_path: str = "/content/drive/My Drive/Academic/research/dissertation/NLP-in-Personality/data/study1/dpo-training-data--main.json"
    model_path: str = "meta-llama/Llama-3.2-3B-Instruct"
    eval_data_path: str | None = "/content/drive/My Drive/Academic/research/dissertation/NLP-in-Personality/data/study1/dpo-eval-data--main.json"
    system_prompt: str | None = "You are a helpful AI assistant."
    model_kwargs: dict[str, any] = field(default_factory=dict)
    tokenizer_kwargs: dict[str, any] = field(default_factory=dict)
    dpo_kwargs: dict[str, any] = field(default_factory=dict)
    shuffle_training_data: bool = True
    random_seed: int = 42
    train_mode: bool = True
    output_dir: str = "/content/drive/MyDrive/Colab Notebooks/dissertation/study1/ouput-dpo"

    @classmethod
    def from_trained(cls, model_path: str, **kwargs):
        return cls(model_path=model_path, train_mode=False, **kwargs)

    # private methods
    def _reset_obj(self):
        # clear gpu memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        return None

    def _offload_obj(self):
        if not hasattr(self, '_dpo_trainer'):
            del self._dpo_trainer
            del self._model
            del self._tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    def _setup_model(self):
        self._model = AutoModelForCausalLM.from_pretrained(self.model_path, **self.get_model_kwargs)

    def _setup_tokenizer(self):
        self._tokenizer = AutoTokenizer.from_pretrained(self.model_path, **self.get_tokenizer_kwargs)
        self._tokenizer.pad_token = self._tokenizer.eos_token
        self._tokenizer.padding_side = "left"
        self._tokenizer.truncation_side = "left"

    def _load_data(self):
        if not os.path.exists(self.data_path):
            from google.colab import drive
            drive.mount('/content/drive')
        if isinstance(self.data_path, str) & os.path.exists(self.data_path):
            _data_paths =  {"train": self.data_path}
            if self.eval_data_path:
                _data_paths["eval"] = self.eval_data_path
            self._data = load_dataset("json", data_files=_data_paths)
            self._data_column_names = self._data["train"].column_names
            if self.shuffle_training_data:
                self._data = self._data.shuffle(seed = self.random_seed)
        else:
            raise NotImplementedError("data_path must be valid path to json file.")

    def _setup_preference_data(self):
        self.create_preference_data()

    def _setup_dpo(self):
        self._dpo_trainer = DPOTrainer(**self.get_dpo_kwargs)

    def _setup_pipeline(self):
        # Load Model with PEFT adapter
        self._infer_model = AutoPeftModelForCausalLM.from_pretrained(self.dpo_model_path,
                                                                     device_map = "auto",
                                                                     torch_dtype = torch.bfloat16)
        self._infer_tokenizer = AutoTokenizer.from_pretrained(self.dpo_model_path)
        # load into pipeline
        self._infer_pipe = pipeline("text-generation", model = self._infer_model, tokenizer = self._infer_tokenizer)

    ## Post-Init
    def __post_init__(self):
        if self.train_mode:
            self._reset_obj()
            self._load_data()
            self._setup_model()
            self._setup_tokenizer()
            self._setup_preference_data()
            self._setup_dpo()
        else:
            self.dpo_model_path = self.output_dir
            self._setup_pipeline()

    ## Train Function
    def train(self, save_model_at_end: bool = True, train_kwargs: dict = {}, output_kwargs: dict  = {}, **kwargs):
        if not hasattr(self, '_dpo_trainer'):
            self._setup_dpo()
        self._reset_obj() # for gpu memory purposes
        _train_kwargs = train_kwargs | {**kwargs}
        self._dpo_trainer.train(**_train_kwargs)
        if save_model_at_end:
            self.save(**output_kwargs)
        return self._dpo_trainer

    ## Save Function
    def save(self, **output_kwargs):
        self._dpo_trainer.save_model(**output_kwargs)
        self._offload_obj()
        self.load_merge_and_unload()
        self._setup_pipeline()

    ## Inference Function
    def infer(self, prompts: list[str] | str, add_system_prompt: bool = True, inference_kwargs: dict | None = None, see_prompts: bool = False):
        if not hasattr(self, '_infer_pipe'):
            self._setup_pipeline()
        def _as_message(_prompt: str, system_prompt: bool) -> list[dict]:
            return [{"role": "system", "content": self.system_prompt}, {"role":"user", "content": _prompt}] if system_prompt else [{"role":"user", "content": _prompt}]
        _INFER_MSGS = []
        if isinstance(prompts, str):
            prompts = [prompts]
        for prompt in prompts:
            _INFER_MSGS.append(self._infer_pipe.tokenizer.apply_chat_template(_as_message(prompt, add_system_prompt), tokenize = False, add_generation_prompt = True))
        if see_prompts:
            return _INFER_MSGS
        if not inference_kwargs:
            inference_kwargs = self.get_infer_kwargs
        return [self._infer_pipe(_MSG, **inference_kwargs) for _MSG in _INFER_MSGS]

    ## Utility Functions
    @staticmethod
    def _to_preference_data(example) -> dict[str,str]:
        """Transforms a single example into preference data format."""
        return {
            "prompt": example['prompt'],
            "chosen": example['chosen'],
            "rejected": example['rejected'],
            }

    def create_preference_data(self, data: Dataset | None = None) -> None:
        """
        Transforms a dataset into a preference dataset format.
        """
        if not hasattr(self, '_data'):
            raise AttributeError("Data has yet to be imported. Please try again")
        self._preference_data = self._data.map(self._to_preference_data, remove_columns=self._data_column_names)
        return None

    def load_merge_and_unload(self, out_path: str | None = None):
        _model_path = self.default_dpo_training_kwargs.get('output_dir', os.getcwd() + "/")
        _peft_base = AutoPeftModelForCausalLM.from_pretrained(_model_path, device_map="auto", torch_dtype = torch.bfloat16)
        # Merge LoRA and base model
        merged_model = _peft_base.merge_and_unload()
        # Save merged model
        if not out_path:
            out_path = _model_path
        merged_model.save_pretrained(out_path, safe_serialization = True, max_shard_size = "2GB")
        del merged_model
        del _peft_base
        gc.collect()
        torch.cuda.empty_cache()
        self.dpo_model_path = out_path

    ## Properties
    @property
    def has_eval_data(self) -> bool:
        return self._data.get("eval", None) is not None

    @property
    def get_dpo_kwargs(self) -> dict[str, any]:
        return self.default_dpo_kwargs | self.dpo_kwargs

    @property
    def get_dpo_training_kwargs(self) -> dict[str, any]:
        return self.default_dpo_training_kwargs | self.default_dpo_training_eval_kwargs

    @property
    def get_model_kwargs(self) -> dict[str, any]:
        return self.default_model_kwargs | self.model_kwargs

    @property
    def get_tokenizer_kwargs(self) -> dict[str, any]:
        return self.tokenizer_kwargs

    @property
    def get_infer_kwargs(self) -> dict[str, any]:
        return self.default_pipeline_infer_kwargs

    @property
    def default_dpo_kwargs(self) -> dict[str, any]:
        return {"model": self._model,
                "train_dataset": self._preference_data["train"],
                "eval_dataset": self._preference_data.get("eval", None) ,
                "peft_config": LoraConfig(**self.default_peft_kwargs),
                "processing_class": self._tokenizer,
                "args": DPOConfig(**self.get_dpo_training_kwargs),
                }

    @property
    def default_peft_kwargs(self) -> dict[str, any]:
        return {"r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
                "bias": "none", "task_type": "CAUSAL_LM",
                "target_modules": "all-linear",}

    @property
    def default_model_kwargs(self) -> dict[str, any]:
        # TODO -- debug "attn_implementation": "flash_attention_2"
        return {"torch_dtype": torch.bfloat16,
                "device_map": "auto",
                "use_cache": False,
                "quantization_config": BitsAndBytesConfig(
                    load_in_4bit=True,
                    llm_int8_threshold=6.0,
                    llm_int8_has_fp16_weight=False,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                    )
                }
    # max prompt size: ~431 token
    # max output size: ~1330 tokens (8 SJT items)
    @property
    def default_dpo_training_kwargs(self) -> dict[str, any]:
        return {"output_dir": self.output_dir,
                "beta": 0.1,
                "loss_type": "sigmoid",
                "optim": "adamw_torch_fused",
                "num_train_epochs": 1,
                "learning_rate": 5e-5,
                "auto_find_batch_size": True,
                "gradient_checkpointing": True,
                "gradient_accumulation_steps": 2,
                "max_grad_norm": 0.3,
                "warmup_ratio": 0.1, "logging_steps": 10,
                "lr_scheduler_type":"cosine",
                "max_length": 1856,
                "max_prompt_length": 460,
                "bf16": True,
                "push_to_hub": False,
                "report_to": "tensorboard",
                "save_total_limit": 1,
                "save_strategy": "no",
                "seed": self.random_seed,
                }

    @property
    def default_dpo_training_eval_kwargs(self) -> dict[str, any]:
        return {"eval_strategy": "steps", "eval_steps": 10, "eval_accumulation_steps": 1, "per_device_eval_batch_size": 4} if self.has_eval_data else {}

    @property
    def default_pipeline_infer_kwargs(self) -> dict[str, any]:
        return {"max_new_tokens": 2048, "temperature": .50, "eos_token_id": [self._infer_tokenizer.eos_token_id, self._infer_tokenizer.convert_tokens_to_ids("<|eot_id|>")], "pad_token_id": self._infer_tokenizer.pad_token_id}

## Train model with DPO

In [ ]:
DPO_TRAIN = DPOobj()
try:
    DPO_TRAIN.train()
except RuntimeError as e:
    print(e)
    total_free_gpu_memory, total_gpu_memory = torch.cuda.mem_get_info()
    print(f"Total free GPU memory: {round(total_free_gpu_memory * 1e-9, 3)} GB")
    print(f"Total GPU memory: {round(total_gpu_memory * 1e-9, 3)} GB")